# Data Analytics and Machine Learning in Finance
## Assignment: Predicting Credit Rating Downgrades with Machine Learning

## How to proceed
This notebook is designed to help you answer the questions in the graded Canvas quiz. To receive full credit, you must upload a fully executed Jupyter notebook with all outputs visible.

## Objective

The goal of this assignment is to predict whether a company will experience a *credit rating downgrade* using firm-level financial characteristics.

Credit rating downgrades are economically important events. In this assignment, downgrade prediction is treated as a binary classification problem using standard machine learning techniques commonly applied in credit risk modeling.

All financial variables in the dataset are already lagged. This means that the variables only use information that was available before the downgrade decision.

---

## Dataset Description

Each observation corresponds to a company–year.

### Variables

- `company_id`: Unique firm identifier  
- `year`: Fiscal year  
- `downgrade`:  
  - `1` if the firm experienced a credit rating downgrade  
  - `0` otherwise  

### Financial Predictors (All Lagged)

- `debt_to_assets` — leverage  
- `current_ratio` — liquidity  
- `roa` — profitability  
- `interest_coverage` — debt service capacity  
- `log_assets` — firm size  
- `book_to_market` — valuation  

---

## Modeling Task

You should use all listed financial variables as inputs to your machine learning model. The goal is to predict the binary downgrade indicator.

Do not use `company_id` as a predictive feature. This variable is provided only as an identifier.

---

## Data Construction (Background)

The financial ratios are constructed using standard accounting definitions:

- **Leverage**
  - Debt to assets = (long-term debt + short-term debt) / total assets  

- **Liquidity**
  - Current ratio = current assets / current liabilities  

- **Profitability**
  - ROA = net income / total assets  

- **Coverage**
  - Interest coverage = operating income / interest expense  

- **Size**
  - Log assets = ln(total assets)  

- **Valuation**
  - Market value of equity = shares outstanding × stock price  
  - Book-to-market = book equity / market value of equity  

These variables are widely used in credit risk analysis and closely reflect the information used by rating agencies and financial institutions.

---

## Submission Instructions

Submit: Save a fully executed Jupyter notebook and rename it as your StudentID.
### For example, if your ID was `'1234567'`, your file would be saved as `'1234567.ipynb'`
---

<hr style="border:2px solid teal"> </hr>

###  <span style="color:teal">Write your submission into this solution file</span> 

1. Load the data set (use the `company_id` as the index and the first line as the header) and display the dimensions of the data set.

In [1]:
import pandas as pd

df = pd.read_excel("credit_rating_hw.xlsx", index_col=0)
print(df.shape)


(1877, 8)


2. Display the first 3 rows of the data set.

In [2]:
df.head(3)


,year,downgrade,debt_to_assets,current_ratio,roa,interest_coverage,log_assets,book_to_market
company_id,,,,,,,,
1,2020,0,0.329293,3.755416,0.002116,11.440860,7.639642,1.275027
2,2020,1,0.557446,0.448146,0.028102,3.384475,11.002016,-0.009608
3,2021,1,0.345616,0.880836,0.027500,3.184440,9.904508,0.625291


3. Produce summary statistics of the file.

In [3]:
df.describe()


,year,downgrade,debt_to_assets,current_ratio,roa,interest_coverage,log_assets,book_to_market
count,1877.000000,1877.00000,1877.000000,1877.000000,1877.000000,1877.000000,1877.000000,1877.000000
mean,2019.049547,0.22536,0.438353,1.648966,-0.000972,5.911078,8.532509,0.257468
std,1.410135,0.41793,0.249611,1.106193,0.124795,17.342768,1.477535,4.139998
min,2017.000000,0.00000,0.000000,0.100000,-1.038932,-59.945267,5.359488,-61.095299
25%,2018.000000,0.00000,0.277382,0.961188,-0.029921,0.857167,7.450410,0.169049
50%,2019.000000,0.00000,0.395676,1.394002,0.020558,2.746930,8.395269,0.417520
75%,2020.000000,0.00000,0.548676,1.992651,0.056863,5.792502,9.503329,0.814980
max,2021.000000,1.00000,1.929458,9.144014,0.325871,215.718552,12.679804,13.800114


4. Display the unique values of the variable `year`.

In [4]:
sorted(df["year"].unique())


[2017, 2018, 2019, 2020, 2021]

5. Multiple choice question 

6. Split the data set into train and test data set (approximately in the ratio 80:20) in a way that economically makes sense. Create X_train, y_train, X_test, y_test. Do not include the variable year neither in X_train, nor in X_test.

In [5]:
import numpy as np

years = sorted(df["year"].unique())
split_idx = int(len(years) * 0.8)
train_years = years[:split_idx]
test_years = years[split_idx:]

train_df = df[df["year"].isin(train_years)]
test_df = df[df["year"].isin(test_years)]

X_train = train_df.drop(columns=["downgrade", "year"])
y_train = train_df["downgrade"]
X_test = test_df.drop(columns=["downgrade", "year"])
y_test = test_df["downgrade"]

print(f"Train years: {train_years}")
print(f"Test years: {test_years}")
print(X_train.shape, X_test.shape)


Train years: [2017, 2018, 2019, 2020]
Test years: [2021]
(1516, 6) (361, 6)


For the following three questions, print the performance on the train set and the test set rounded to two decimals:
7. Run a logistic regression with elastic net with l1_ratio of 0.7 and maximum number of iterations set to 4000.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_en = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    l1_ratio=0.7,
    max_iter=4000,
)
log_en.fit(X_train, y_train)

train_acc = accuracy_score(y_train, log_en.predict(X_train))
test_acc = accuracy_score(y_test, log_en.predict(X_test))

print(f"Train accuracy: {train_acc:.2f}")
print(f"Test accuracy: {test_acc:.2f}")


Train accuracy: 0.75
Test accuracy: 0.85


/Users/dennissydow/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


8. Estimate a logistic regression model with L1 regularization with solver= liblinear.

Identify the three variables with the largest absolute coefficients.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_l1 = LogisticRegression(penalty="l1", solver="liblinear", max_iter=4000)
log_l1.fit(X_train, y_train)

train_acc = accuracy_score(y_train, log_l1.predict(X_train))
test_acc = accuracy_score(y_test, log_l1.predict(X_test))

print(f"Train accuracy: {train_acc:.2f}")
print(f"Test accuracy: {test_acc:.2f}")

coef = pd.Series(log_l1.coef_[0], index=X_train.columns)
top3 = coef.abs().sort_values(ascending=False).head(3)
print("Top 3 variables by |coef|:")
print(top3.index.tolist())


Train accuracy: 0.76
Test accuracy: 0.83
Top 3 variables by |coef|:
['roa', 'debt_to_assets', 'current_ratio']


9. Run a random forest model with 100 trees and with a maximum depth of 3. Also, set the `random_state` to 37.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=37)
rf.fit(X_train, y_train)

train_acc = accuracy_score(y_train, rf.predict(X_train))
test_acc = accuracy_score(y_test, rf.predict(X_test))

print(f"Train accuracy: {train_acc:.2f}")
print(f"Test accuracy: {test_acc:.2f}")


Train accuracy: 0.77
Test accuracy: 0.85


10. Run a gradient boosting model with 300 boosting stages and the same random state as in the previous question.

In [9]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

gb = GradientBoostingClassifier(n_estimators=300, random_state=37)
gb.fit(X_train, y_train)

train_acc = accuracy_score(y_train, gb.predict(X_train))
test_acc = accuracy_score(y_test, gb.predict(X_test))

print(f"Train accuracy: {train_acc:.2f}")
print(f"Test accuracy: {test_acc:.2f}")


Train accuracy: 0.92
Test accuracy: 0.80


11. Fine-tune the random forest model from the question above by varying the following hyperparameters:

max_depth ∈ {2, 3, 4, 5}

n_estimators ∈ {100, 200, 300}

Use 5-fold cross-validation with shuffling on the training set.
Select the model that achieves the highest average accuracy across the cross-validation folds.

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=37)

best_score = -1
best_params = None

for max_depth in [2, 3, 4, 5]:
    for n_estimators in [100, 200, 300]:
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=37,
        )
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
        mean_score = scores.mean()
        if mean_score > best_score:
            best_score = mean_score
            best_params = {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
            }

print(f"Best params: {best_params}")
print(f"Best CV accuracy: {best_score:.2f}")

best_rf = RandomForestClassifier(**best_params, random_state=37)
best_rf.fit(X_train, y_train)

print(f"Train accuracy: {best_rf.score(X_train, y_train):.2f}")
print(f"Test accuracy: {best_rf.score(X_test, y_test):.2f}")


Best params: {'n_estimators': 200, 'max_depth': 3}
Best CV accuracy: 0.76
Train accuracy: 0.77
Test accuracy: 0.86


12. Fine-tune the gradient boosting model from the question above by varying the number of boosting stages as follows:

n_estimators ∈ {100, 200, 300, 400, 500}

Use 5-fold cross-validation with shuffling on the training set.
Select the model that achieves the highest average accuracy across the cross-validation folds.

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=37)

best_score = -1
best_n_estimators = None

for n_estimators in [100, 200, 300, 400, 500]:
    model = GradientBoostingClassifier(n_estimators=n_estimators, random_state=37)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    mean_score = scores.mean()
    if mean_score > best_score:
        best_score = mean_score
        best_n_estimators = n_estimators

print(f"Best n_estimators: {best_n_estimators}")
print(f"Best CV accuracy: {best_score:.2f}")

best_gb = GradientBoostingClassifier(n_estimators=best_n_estimators, random_state=37)
best_gb.fit(X_train, y_train)

print(f"Train accuracy: {best_gb.score(X_train, y_train):.2f}")
print(f"Test accuracy: {best_gb.score(X_test, y_test):.2f}")


Best n_estimators: 100
Best CV accuracy: 0.74
Train accuracy: 0.84
Test accuracy: 0.80


<hr style="border:2px solid teal"> </hr>

###  <span style="color:teal">Instructions for submitting the file</span> 

Save the Jupyter Notebook and rename it as your StudentID.
For example, if your ID was `'1234567'`, your file would be saved as `'1234567.ipynb'`

© Copyright. 2026. Prof. Dr. Kornelia Fabisik.

<hr style="border:2px solid gray"> </hr>